# PhysIR — Stage 3 Sweep · ACCOUNT C (extension, HANDS-OFF)

**Grid winner was `ph12_re18` (25.27) — a corner optimum**, so this extends the sweep to
find the true peak: push phase up (retinex fixed at 0.18) + one retinex-edge check.

| config | phase | retinex | purpose |
|--------|------:|--------:|---------|
| ph16_re18 | 0.16 | 0.18 | phase ceiling probe |
| ph20_re18 | 0.20 | 0.18 | phase ceiling probe |
| ph12_re25 | 0.12 | 0.25 | retinex edge check |

**Hands-off:** one loop runs all three back-to-back — no per-config babysitting. It's
**idempotent + resume-safe**: completed configs (30 ep) are skipped, and a partially-trained
config auto-resumes from its checkpoint. So if the Kaggle session times out, just **re-run
the notebook** (or re-commit via *Save & Run All*) and it continues where it left off.

> ⏱️ 3 configs × ~3.5–4 hr ≈ 10.5–12 hr — may exceed one session. That's fine: re-run to
> finish. To stay inside one session, trim `RUN_LIST` to 2 and run the third later.

**Env (matches Stage 2/3 runs):** code + data are Kaggle input datasets (read-only, no clone);
cwd stays `/kaggle/working` so `.metrics` lands there.  **Accelerator:** GPU T4 × 2.

> ⚠️ Blur is OFF for all three (grid uses no blur), so the resume σ-reset bug is irrelevant
> here — retinex/phase have no learnable params, resume is fully safe.

## 1. Install dependencies

In [ ]:
!pip install -q lpips pytorch-msssim einops ptflops pyiqa

## 2. Paths — edit before running

In [ ]:
import os

# ── EDIT THESE (Kaggle input datasets — read-only) ────────────────────────────
CODE_DIR     = '/kaggle/input/datasets/jeffreyboahen/physir-code'      # repo as a dataset
LOLBLUR_ROOT = '/kaggle/input/datasets/jeffreyboahen/lolblur-dataset'  # LOLBlur
CHECKPOINT_DIR = '/kaggle/working'
# ──────────────────────────────────────────────────────────────────────────────

WORKING_DIR = '/kaggle/working'   # cwd stays here → .metrics writes to /kaggle/working/.metrics
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
assert os.path.isdir(CODE_DIR), f'CODE_DIR not found: {CODE_DIR}'
assert os.path.isfile(os.path.join(CODE_DIR, 'train.py')), 'train.py not in CODE_DIR'
print(f'Code:        {CODE_DIR}')
print(f'Dataset:     {LOLBLUR_ROOT}')
print(f'Checkpoints: {CHECKPOINT_DIR}')

## 3. Train/val split (same 90/10, seed 42 — identical to grid)

In [ ]:
import random
from pathlib import Path

SUBS       = ['low_blur_noise', 'high_sharp_scaled']
TRAIN_SRC  = Path(LOLBLUR_ROOT) / 'train'
SPLIT_ROOT = Path(WORKING_DIR) / 'LOLBlur_split'

random.seed(42)
clips = sorted([d.name for d in (TRAIN_SRC / 'low_blur_noise').iterdir() if d.is_dir()])
n_val = max(1, int(len(clips) * 0.1))
val_clips   = set(random.sample(clips, n_val))
train_clips = [c for c in clips if c not in val_clips]
print(f'Clips — train: {len(train_clips)}, val: {len(val_clips)}')

for split_name, clip_list in [('physir_train', train_clips), ('physir_val', val_clips)]:
    for sub in SUBS:
        dst_base = SPLIT_ROOT / split_name / sub
        dst_base.mkdir(parents=True, exist_ok=True)
        for clip in clip_list:
            src_path = (TRAIN_SRC / sub / clip).resolve()
            dst_path = dst_base / clip
            if not dst_path.exists():
                dst_path.symlink_to(src_path)

PHYSIR_TRAIN = str(SPLIT_ROOT / 'physir_train')
PHYSIR_VAL   = str(SPLIT_ROOT / 'physir_val')
print(f'  train: {PHYSIR_TRAIN}')
print(f'  val:   {PHYSIR_VAL}')

## 4. Hands-off batch — runs all Account C configs sequentially

Idempotent: skips configs already at 30 ep, resumes partial ones. Re-run freely.

In [ ]:
import yaml, csv, subprocess
from pathlib import Path

EPOCHS = 30

# ── Account C extension configs (blur OFF) ────────────────────────────────────
SWEEP = {
    'sweep_ph16_re18': {'phase': 0.16, 'retinex': 0.18, 'blur': 0.0},
    'sweep_ph20_re18': {'phase': 0.20, 'retinex': 0.18, 'blur': 0.0},
    'sweep_ph12_re25': {'phase': 0.12, 'retinex': 0.25, 'blur': 0.0},
}
# Priority order: the two PHASE-CEILING probes run FIRST (most informative — they test
# whether pushing phase past 0.12 keeps winning). The retinex-edge check runs last.
# If a session can only fit 2, the important pair is already done.
PRIORITY = ['sweep_ph16_re18', 'sweep_ph20_re18']   # run first
SECONDARY = ['sweep_ph12_re25']                     # run last
RUN_LIST = PRIORITY + SECONDARY
# ──────────────────────────────────────────────────────────────────────────────

def build_cfg(run_name, sel):
    blur_on = sel['blur'] > 0
    cfg = {
        'datasets': {
            'name': 'LOLBlur',
            'train': {'low_path': f'{PHYSIR_TRAIN}/low_blur_noise',
                      'high_path': f'{PHYSIR_TRAIN}/high_sharp_scaled',
                      'crop_size': 256, 'batch_size': 4, 'n_workers': 4, 'use_flips': True},
            'val': {'test_path': PHYSIR_VAL, 'batch_size_test': 1,
                    'verbose': False, 'n_workers': 4},
        },
        'network': {'name': 'DarkIR', 'img_channels': 3, 'width': 32,
                    'middle_blk_num_enc': 2, 'middle_blk_num_dec': 2,
                    'enc_blk_nums': [1, 2, 3], 'dec_blk_nums': [3, 1, 1],
                    'dilations': [1, 4, 9], 'extra_depth_wise': True},
        'save': {'pretrained': None,
                 'path': f'{CHECKPOINT_DIR}/{run_name}_latest.pt',
                 'best': f'{CHECKPOINT_DIR}/{run_name}_best.pt',
                 'save_freq': 5, 'eval_freq': 5, 'perceptual_eval_epochs': []},
        'pixel_criterion': 'l1', 'perceptual': False,
        'edge': True, 'edge_weight': 0.05, 'edge_criterion': 'l2', 'edge_reduction': 'mean',
        'frequency': False,
        'enhance': True, 'enhance_weight': 0.01,
        'enhance_criterion': 'l1', 'enhance_reduction': 'mean',
        'physics_losses': {
            'enabled': True,
            'phase2_start': 9999, 'phase3_start': 9999, 'phase4_start': 1,
            'retinex': True, 'retinex_weight': sel['retinex'],
            'blur': blur_on, 'blur_weight': sel['blur'],
            'phase': True, 'phase_weight': sel['phase'],
        },
        'train': {'epochs': EPOCHS, 'lr_initial': 0.00028, 'weight_decay': 0.001,
                  'lr_scheme': 'CosineAnnealing', 'eta_min': 1e-6, 'grad_clip': 1.0},
        'wandb': {'init': False, 'name': run_name},
        'Resize': False,
    }
    cfg_path = f'{WORKING_DIR}/{run_name}.yml'
    with open(cfg_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)
    return cfg_path

def done_epoch(run_name):
    log = Path(WORKING_DIR) / '.metrics' / run_name / 'training_log.csv'
    if not log.exists():
        return 0
    last = 0
    try:
        with open(log) as f:
            for r in csv.DictReader(f):
                last = max(last, int(r['epoch']))
    except Exception:
        pass
    return last

print(f'Account C queue (priority first): {RUN_LIST}\n')
for run_name in RUN_LIST:
    sel = SWEEP[run_name]
    d = done_epoch(run_name)
    if d >= EPOCHS:
        print(f'✅ {run_name} already at {d}/{EPOCHS} ep — skipping.\n')
        continue
    tag = f'(resuming from ep {d})' if d > 0 else '(from scratch)'
    print(f'===== {run_name}  phase={sel["phase"]} retinex={sel["retinex"]}  {tag} =====')
    cfg_path = build_cfg(run_name, sel)
    try:
        r = subprocess.run(['torchrun', '--nproc_per_node=2',
                            f'{CODE_DIR}/train.py', '-p', cfg_path])
        print(f'===== {run_name} finished (exit={r.returncode}) =====\n')
    except Exception as e:
        print(f'===== {run_name} FAILED: {e} — continuing =====\n')

print('All Account C configs processed. Run the leaderboard cell to see standings.')

## 5. Leaderboard — all sweep configs under WORKING_DIR/.metrics

For the FULL ranking (grid + extension), download this account's `.metrics/sweep_*` and
combine with the grid dirs from accounts A/B before picking the final winner.

In [ ]:
import pandas as pd
from pathlib import Path

metrics_root = Path(WORKING_DIR) / '.metrics'
rows = []
for d in sorted(metrics_root.glob('sweep_*')):
    log = d / 'training_log.csv'
    if not log.exists():
        continue
    df = pd.read_csv(log)
    df.replace('', float('nan'), inplace=True)
    val = df.dropna(subset=['val_psnr']) if 'val_psnr' in df.columns else df.iloc[0:0]
    if not len(val):
        rows.append({'config': d.name, 'best_psnr': None, 'epochs': int(df['epoch'].max())})
        continue
    best = val.loc[val['val_psnr'].idxmax()]
    rows.append({
        'config': d.name,
        'best_psnr': round(float(best['val_psnr']), 4),
        'best_ssim': round(float(best['val_ssim']), 4) if pd.notna(best.get('val_ssim')) else None,
        'best_lpips': round(float(best['val_lpips']), 4) if pd.notna(best.get('val_lpips')) else None,
        'at_epoch': int(best['epoch']),
        'epochs': int(df['epoch'].max()),
    })

if not rows:
    print('No sweep runs found yet under', metrics_root)
else:
    board = pd.DataFrame(rows).sort_values('best_psnr', ascending=False, na_position='last')
    print('=== Sweep Leaderboard (LOLBlur val PSNR @ ≤30 ep) ===')
    print('  Refs:  A=24.24   C phase-only=24.94   Stage2=24.34   grid winner ph12_re18=25.27\n')
    print(board.to_string(index=False))
    done = board[board['epochs'] >= 30]
    if len(done):
        w = done.iloc[0]
        print(f'\n  Leader (≥30 ep): {w["config"]}  →  {w["best_psnr"]} dB')
        print(f'  vs grid winner (25.27): {w["best_psnr"] - 25.266:+.4f} dB')